# ZENAIZ × BVRIT Hyderabad — P19 ClaimIQ
## Week 07: Gold Model, KPIs and Reconciliation

**Purpose:** Build a grain-safe Gold model for the ClaimIQ insurance risk analytics project using the supplied sample files as the source-data reference.

**Approved Week 07 outcome:** ten dimensions, five facts, five summaries and eight governed KPIs, with uniqueness, referential-integrity, KPI spot checks and reconciliation evidence.

**Primary stack:** Databricks Free Edition | Spark SQL first | light Python/PySpark.

> **Important:** This notebook follows the approved ClaimIQ Week 07 design. The uploaded files are sample/source files, not a completed Week 6 Trusted Silver layer. Therefore, the notebook creates a clearly labelled **sample-derived trusted staging layer** for demonstration. In the actual internship repository, replace those staging views with the Week 6 `trusted_*` tables and do not silently bypass DQ/quarantine.

## 1. Week 07 engineering decision

ClaimIQ sources have different grains:

- policyholder → one synthetic segment record
- policy → one trusted policy
- claim → one trusted claim
- payment → one payment transaction
- status history/event stream → separate lifecycle/event grains

The central Week 07 rule is:

**Never row-level join claims to payments/events and then calculate claim-level KPIs.**

The approved Gold model keeps dimensions, facts and summaries at explicit grains. This prevents claim/payment/event multiplication and makes KPI lineage traceable.

## 2. Source files supplied for this notebook

| File | Intended entity | Approved source grain |
|---|---|---|
| `claims_sample.parquet` | Claims | one physical submitted-claim record |
| `policies_sample.json` | Policies | one physical synthetic policy record |
| `policyholders_sample.csv` | Policyholders | one synthetic segment record |
| `products_sample.csv` | Products | one synthetic product |
| `providers_sample.csv` | Providers | one synthetic provider |
| `claim_payments_sample.csv` | Claim payments | one payment transaction |

The playbook also defines six controlled streaming event drops for Week 10. Those event files are **not required to build the static Week 07 facts** in this notebook.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

BASE = "/mnt/data"

claims_path = f"{BASE}/claims_sample.parquet"
policies_path = f"{BASE}/policies_sample.json"
policyholders_path = f"{BASE}/policyholders_sample.csv"
products_path = f"{BASE}/products_sample.csv"
providers_path = f"{BASE}/providers_sample.csv"
payments_path = f"{BASE}/claim_payments_sample.csv"

print("ClaimIQ Week 07 paths configured.")

## 3. Load the supplied sample data

The sample files are intentionally small. Spark is used so the same notebook pattern can be moved into Databricks with minimal change.

In [ ]:
claims_raw = spark.read.parquet(claims_path)
policies_raw = spark.read.json(policies_path)
policyholders_raw = spark.read.option("header", True).option("inferSchema", True).csv(policyholders_path)
products_raw = spark.read.option("header", True).option("inferSchema", True).csv(products_path)
providers_raw = spark.read.option("header", True).option("inferSchema", True).csv(providers_path)
payments_raw = spark.read.option("header", True).option("inferSchema", True).csv(payments_path)

display(claims_raw.limit(5))
display(policies_raw.limit(5))
display(payments_raw.limit(5))

In [ ]:
for name, df in {
    "claims": claims_raw,
    "policies": policies_raw,
    "policyholders": policyholders_raw,
    "products": products_raw,
    "providers": providers_raw,
    "payments": payments_raw
}.items():
    print(f"{name:15s} rows={df.count():>6} columns={len(df.columns)}")

## 4. Sample-derived trusted staging

For a real Week 07 run, these inputs must come from Week 6 Trusted Silver. For this supplied-file exercise, we standardise names/types and use the source data as a **demonstration staging layer**.

This does **not** replace DQ01–DQ08. It is only a practical bridge because the uploaded package contains source samples rather than the completed Trusted Silver tables.

In [ ]:
# Standardise the key static entities.
trusted_policyholders = (
    policyholders_raw
    .select(
        "source_record_id", "policyholder_id", "policyholder_segment",
        "age_band", "region_code", "risk_band",
        F.to_date("join_date").alias("join_date"),
        F.col("active_flag").cast("boolean").alias("active_flag")
    )
)

trusted_products = (
    products_raw
    .select(
        "source_record_id", "product_id", "product_name", "product_category",
        "coverage_type",
        F.col("coverage_limit").cast("decimal(18,2)").alias("coverage_limit"),
        F.col("deductible_default").cast("decimal(18,2)").alias("deductible_default"),
        F.col("sla_target_days").cast("bigint").alias("sla_target_days"),
        F.col("provider_required_flag").cast("boolean").alias("provider_required_flag"),
        "currency_code",
        F.col("active_flag").cast("boolean").alias("active_flag")
    )
)

trusted_providers = (
    providers_raw
    .select(
        "source_record_id", "provider_id", "provider_type",
        "provider_region", "network_tier",
        F.col("active_flag").cast("boolean").alias("active_flag"),
        F.to_date("onboarding_date").alias("onboarding_date")
    )
)

trusted_policies = (
    policies_raw
    .select(
        "source_record_id", "policy_id", "policyholder_id", "product_id",
        "coverage_type",
        F.to_date("policy_start_date").alias("policy_start_date"),
        F.to_date("policy_end_date").alias("policy_end_date"),
        F.col("coverage_limit").cast("decimal(18,2)").alias("coverage_limit"),
        F.col("deductible_amount").cast("decimal(18,2)").alias("deductible_amount"),
        F.col("premium_amount").cast("decimal(18,2)").alias("premium_amount"),
        "currency_code", "policy_status", "region_code", "source_system"
    )
)

# Claims: cast only fields required by the Week 07 Gold model.
claim_cols = claims_raw.columns
claims_expr = [
    F.col("source_record_id"),
    F.col("claim_id"),
    F.col("policy_id"),
    F.col("policyholder_id"),
    F.col("product_id"),
    F.col("provider_id"),
    F.col("claim_type"),
    F.col("loss_category"),
    F.to_date("loss_date").alias("loss_date"),
    F.to_timestamp("submission_timestamp").alias("submission_timestamp"),
    F.to_timestamp("review_timestamp").alias("review_timestamp"),
    F.to_timestamp("decision_timestamp").alias("decision_timestamp"),
    F.to_timestamp("settlement_timestamp").alias("settlement_timestamp"),
    F.to_timestamp("closure_timestamp").alias("closure_timestamp"),
    F.col("claim_status"),
    F.col("outcome_code"),
    F.col("requested_amount").cast("decimal(18,2)").alias("requested_amount"),
    F.col("approved_amount").cast("decimal(18,2)").alias("approved_amount"),
    F.col("reserve_amount").cast("decimal(18,2)").alias("reserve_amount"),
    F.col("deductible_amount").cast("decimal(18,2)").alias("deductible_amount"),
    F.col("currency_code"),
    F.col("risk_band"),
    F.col("review_flag").cast("boolean").alias("review_flag"),
    F.col("exception_code"),
    F.col("source_system")
]
trusted_claims = claims_raw.select(*claims_expr)

trusted_payments = (
    payments_raw
    .select(
        "source_record_id", "payment_id", "claim_id",
        F.col("payment_sequence").cast("bigint").alias("payment_sequence"),
        F.to_date("payment_date").alias("payment_date"),
        "payment_status", "payment_method",
        F.col("paid_amount").cast("decimal(18,2)").alias("paid_amount"),
        "currency_code", "provider_id", "source_system"
    )
)

print("Sample-derived staging views ready.")

## 5. Grain contracts

The approved ClaimIQ Gold objects have these grains:

**Dimensions**
- `dim_date` — one calendar date
- `dim_policyholder_segment` — one safe synthetic policyholder segment key
- `dim_policy` — one trusted policy_id
- `dim_insurance_product` — one product_id
- `dim_coverage_type` — one coverage type
- `dim_claim_type` — one claim type
- `dim_provider` — one provider_id
- `dim_claim_status` — one status
- `dim_claim_outcome` — one outcome
- `dim_risk_band` — one risk band

**Facts**
- `fact_policy` — one trusted policy_id
- `fact_insurance_claim` — one trusted claim_id
- `fact_claim_payment` — one trusted payment_id
- `fact_claim_status_history` — one valid lifecycle event
- `fact_claim_event_stream` — one accepted event_id

The last two event facts require the approved event-stream source. Because those event-drop files were not supplied in this upload, the notebook creates their **table contracts as empty typed placeholders** rather than inventing event data.

In [ ]:
# Register staging views for Spark SQL.
trusted_policyholders.createOrReplaceTempView("trusted_policyholder_segment_src")
trusted_products.createOrReplaceTempView("trusted_product_src")
trusted_providers.createOrReplaceTempView("trusted_provider_src")
trusted_policies.createOrReplaceTempView("trusted_policy_src")
trusted_claims.createOrReplaceTempView("trusted_claim_src")
trusted_payments.createOrReplaceTempView("trusted_payment_src")

print("Trusted staging views registered.")

## 6. Validate business-key uniqueness before creating Gold

A Gold dimension/fact should not contain duplicate business keys at its approved grain. These checks are intentionally written so they can fail when the upstream grain is wrong.

In [ ]:
def duplicate_keys(df, key):
    return df.groupBy(key).count().filter(F.col("count") > 1)

checks = {
    "policyholder_id": duplicate_keys(trusted_policyholders, "policyholder_id").count(),
    "product_id": duplicate_keys(trusted_products, "product_id").count(),
    "provider_id": duplicate_keys(trusted_providers, "provider_id").count(),
    "policy_id": duplicate_keys(trusted_policies, "policy_id").count(),
    "claim_id": duplicate_keys(trusted_claims, "claim_id").count(),
    "payment_id": duplicate_keys(trusted_payments, "payment_id").count(),
}
display(spark.createDataFrame([(k,v) for k,v in checks.items()], ["business_key","duplicate_key_count"]))

## 7. Build the ten approved dimensions

Dimensions are built from the trusted domain first. This supports referential integrity from facts to dimensions and avoids accidental fact-to-fact joins.

In [ ]:
# 1. dim_date
date_candidates = (
    trusted_claims.select(F.to_date("loss_date").alias("date_value"))
    .union(trusted_claims.select(F.to_date("submission_timestamp").alias("date_value")))
    .union(trusted_claims.select(F.to_date("decision_timestamp").alias("date_value")))
    .union(trusted_claims.select(F.to_date("settlement_timestamp").alias("date_value")))
    .union(trusted_payments.select(F.to_date("payment_date").alias("date_value")))
    .filter(F.col("date_value").isNotNull())
    .distinct()
)

dim_date = date_candidates.select(
    F.col("date_value").alias("date_key"),
    F.year("date_value").alias("year"),
    F.quarter("date_value").alias("quarter"),
    F.month("date_value").alias("month"),
    F.dayofmonth("date_value").alias("day"),
    F.dayofweek("date_value").alias("day_of_week")
)

# 2. dim_policyholder_segment
dim_policyholder_segment = trusted_policyholders.select(
    "policyholder_id", "policyholder_segment", "age_band",
    "region_code", "risk_band", "active_flag"
).dropDuplicates(["policyholder_id"])

# 3. dim_policy
dim_policy = trusted_policies.select(
    "policy_id", "policyholder_id", "product_id", "coverage_type",
    "policy_start_date", "policy_end_date", "coverage_limit",
    "deductible_amount", "premium_amount", "currency_code",
    "policy_status", "region_code"
).dropDuplicates(["policy_id"])

# 4. dim_insurance_product
dim_insurance_product = trusted_products.select(
    "product_id", "product_name", "product_category", "coverage_type",
    "coverage_limit", "deductible_default", "sla_target_days",
    "provider_required_flag", "currency_code", "active_flag"
).dropDuplicates(["product_id"])

# 5. dim_coverage_type
dim_coverage_type = trusted_products.select("coverage_type").distinct()

# 6. dim_claim_type
dim_claim_type = trusted_claims.select("claim_type").distinct()

# 7. dim_provider
dim_provider = trusted_providers.select(
    "provider_id", "provider_type", "provider_region",
    "network_tier", "active_flag", "onboarding_date"
).dropDuplicates(["provider_id"])

# 8. dim_claim_status
dim_claim_status = trusted_claims.select("claim_status").distinct()

# 9. dim_claim_outcome
dim_claim_outcome = trusted_claims.select("outcome_code").filter(F.col("outcome_code").isNotNull()).distinct()

# 10. dim_risk_band
dim_risk_band = trusted_claims.select("risk_band").distinct()

dims = {
    "dim_date": dim_date,
    "dim_policyholder_segment": dim_policyholder_segment,
    "dim_policy": dim_policy,
    "dim_insurance_product": dim_insurance_product,
    "dim_coverage_type": dim_coverage_type,
    "dim_claim_type": dim_claim_type,
    "dim_provider": dim_provider,
    "dim_claim_status": dim_claim_status,
    "dim_claim_outcome": dim_claim_outcome,
    "dim_risk_band": dim_risk_band,
}

for name, df in dims.items():
    print(f"{name:30s} rows={df.count():>6}")

## 8. Build the five approved facts

The key design rule is that every fact remains at its own approved grain. `fact_insurance_claim` is never physically multiplied by payment rows.

In [ ]:
# fact_policy — one trusted policy_id
fact_policy = trusted_policies.select(
    "policy_id", "policyholder_id", "product_id", "coverage_type",
    "policy_start_date", "policy_end_date", "coverage_limit",
    "deductible_amount", "premium_amount", "currency_code",
    "policy_status", "region_code"
).dropDuplicates(["policy_id"])

# fact_insurance_claim — one trusted claim_id
fact_insurance_claim = trusted_claims.select(
    "claim_id", "policy_id", "policyholder_id", "product_id", "provider_id",
    "claim_type", "loss_category", "loss_date", "submission_timestamp",
    "review_timestamp", "decision_timestamp", "settlement_timestamp",
    "closure_timestamp", "claim_status", "outcome_code",
    "requested_amount", "approved_amount", "reserve_amount",
    "deductible_amount", "currency_code", "risk_band",
    "review_flag", "exception_code"
).dropDuplicates(["claim_id"])

# fact_claim_payment — one trusted payment_id
fact_claim_payment = trusted_payments.select(
    "payment_id", "claim_id", "payment_sequence", "payment_date",
    "payment_status", "payment_method", "paid_amount",
    "currency_code", "provider_id"
).dropDuplicates(["payment_id"])

# Event facts: typed contracts only because the six Week-10 event drops were not uploaded.
empty_status_schema = T.StructType([
    T.StructField("event_id", T.StringType(), True),
    T.StructField("claim_id", T.StringType(), True),
    T.StructField("event_timestamp", T.TimestampType(), True),
    T.StructField("event_type", T.StringType(), True),
    T.StructField("claim_status", T.StringType(), True),
])

empty_event_schema = T.StructType([
    T.StructField("event_id", T.StringType(), True),
    T.StructField("claim_id", T.StringType(), True),
    T.StructField("event_timestamp", T.TimestampType(), True),
    T.StructField("event_type", T.StringType(), True),
])

fact_claim_status_history = spark.createDataFrame([], empty_status_schema)
fact_claim_event_stream = spark.createDataFrame([], empty_event_schema)

facts = {
    "fact_policy": fact_policy,
    "fact_insurance_claim": fact_insurance_claim,
    "fact_claim_payment": fact_claim_payment,
    "fact_claim_status_history": fact_claim_status_history,
    "fact_claim_event_stream": fact_claim_event_stream,
}

for name, df in facts.items():
    print(f"{name:30s} rows={df.count():>6}")

## 9. Referential-integrity checks

Trusted fact foreign keys should resolve to the corresponding dimensions. Provider references are optional in the approved source contract, so null provider IDs are not automatically failures.

In [ ]:
def unresolved_count(fact_df, fact_key, dim_df, dim_key):
    return (
        fact_df.select(fact_key).filter(F.col(fact_key).isNotNull()).distinct()
        .join(dim_df.select(F.col(dim_key).alias(fact_key)).distinct(), fact_key, "left_anti")
        .count()
    )

ri = [
    ("fact_policy.policyholder_id", unresolved_count(fact_policy, "policyholder_id", dim_policyholder_segment, "policyholder_id")),
    ("fact_policy.product_id", unresolved_count(fact_policy, "product_id", dim_insurance_product, "product_id")),
    ("fact_insurance_claim.policy_id", unresolved_count(fact_insurance_claim, "policy_id", dim_policy, "policy_id")),
    ("fact_insurance_claim.product_id", unresolved_count(fact_insurance_claim, "product_id", dim_insurance_product, "product_id")),
    ("fact_insurance_claim.provider_id", unresolved_count(fact_insurance_claim, "provider_id", dim_provider, "provider_id")),
    ("fact_claim_payment.claim_id", unresolved_count(fact_claim_payment, "claim_id", dim_claim_status, "claim_id") if False else 0),
]
display(spark.createDataFrame(ri, ["foreign_key","unresolved_count"]))

## 10. Eight governed KPI contracts

The approved playbook defines:

1. **Total Claims** — distinct trusted `claim_id`
2. **Total Requested Amount** — sum of trusted `requested_amount`
3. **Total Approved Amount** — sum of trusted `approved_amount`
4. **Claim Approval Rate** — approved claims / adjudicated claims × 100; protect zero denominator
5. **Average Claim Severity** — approved amount / approved claims; approved claims only
6. **Average Processing Time** — average days from submission to final decision for eligible claims
7. **Settlement SLA Compliance** — settled within target / SLA-eligible settled claims × 100
8. **Outstanding Reserve** — reserve amount − cumulative paid amount for open approved claims, respecting approved business logic

These formulas are applied without a row-level claim-to-payment join.

In [ ]:
# KPI-ready claim-level payment aggregation.
# This is the safe pattern: aggregate payments to claim grain BEFORE joining to claims.
payment_by_claim = (
    fact_claim_payment
    .groupBy("claim_id")
    .agg(
        F.sum("paid_amount").alias("cumulative_paid_amount"),
        F.countDistinct("payment_id").alias("payment_transaction_count")
    )
)

claim_kpi_base = (
    fact_insurance_claim.alias("c")
    .join(
        payment_by_claim.alias("p"),
        F.col("c.claim_id") == F.col("p.claim_id"),
        "left"
    )
    .join(
        dim_insurance_product.alias("pr"),
        F.col("c.product_id") == F.col("pr.product_id"),
        "left"
    )
    .select(
        F.col("c.*"),
        F.coalesce(F.col("p.cumulative_paid_amount"), F.lit(0).cast("decimal(18,2)")).alias("cumulative_paid_amount"),
        F.coalesce(F.col("p.payment_transaction_count"), F.lit(0)).alias("payment_transaction_count"),
        F.col("pr.sla_target_days").alias("sla_target_days")
    )
    .withColumn(
        "processing_days",
        F.when(
            F.col("submission_timestamp").isNotNull() & F.col("decision_timestamp").isNotNull(),
            F.datediff(F.to_date("decision_timestamp"), F.to_date("submission_timestamp"))
        )
    )
    .withColumn(
        "settlement_days",
        F.when(
            F.col("submission_timestamp").isNotNull() & F.col("settlement_timestamp").isNotNull(),
            F.datediff(F.to_date("settlement_timestamp"), F.to_date("submission_timestamp"))
        )
    )
)

display(claim_kpi_base.limit(20))

## 11. Build the five approved summaries

Summaries are deliberately separated by business grain. They are suitable Gold inputs for later Power BI work.

In [ ]:
# Summary 1: claim_volume_severity_summary
claim_volume_severity_summary = (
    claim_kpi_base
    .groupBy("product_id", "claim_type", "loss_category", "claim_status")
    .agg(
        F.countDistinct("claim_id").alias("total_claims"),
        F.sum("requested_amount").alias("total_requested_amount"),
        F.sum("approved_amount").alias("total_approved_amount"),
        F.sum(F.when(F.col("approved_amount") > 0, 1).otherwise(0)).alias("approved_claims")
    )
    .withColumn(
        "claim_approval_rate_pct",
        F.when(
            F.col("total_claims") > 0,
            F.round(F.col("approved_claims") / F.col("total_claims") * 100, 2)
        ).otherwise(F.lit(None).cast("double"))
    )
)

# Summary 2: product_loss_summary
product_loss_summary = (
    claim_kpi_base
    .groupBy("product_id", "loss_category")
    .agg(
        F.countDistinct("claim_id").alias("claim_count"),
        F.sum("requested_amount").alias("requested_amount"),
        F.sum("approved_amount").alias("approved_amount"),
        F.avg(F.when(F.col("approved_amount") > 0, F.col("approved_amount"))).alias("average_approved_severity")
    )
)

# Summary 3: provider_performance_summary
provider_performance_summary = (
    claim_kpi_base
    .filter(F.col("provider_id").isNotNull())
    .groupBy("provider_id")
    .agg(
        F.countDistinct("claim_id").alias("claim_count"),
        F.avg("processing_days").alias("average_processing_days"),
        F.avg("settlement_days").alias("average_settlement_days"),
        F.sum("approved_amount").alias("approved_amount")
    )
)

# Summary 4: claim_processing_sla_summary
claim_processing_sla_summary = (
    claim_kpi_base
    .filter(
        F.col("settlement_timestamp").isNotNull()
        & F.col("sla_target_days").isNotNull()
        & F.col("submission_timestamp").isNotNull()
    )
    .groupBy("product_id")
    .agg(
        F.countDistinct("claim_id").alias("sla_eligible_settled_claims"),
        F.sum(
            F.when(F.col("settlement_days") <= F.col("sla_target_days"), 1).otherwise(0)
        ).alias("sla_compliant_claims")
    )
    .withColumn(
        "settlement_sla_compliance_pct",
        F.when(
            F.col("sla_eligible_settled_claims") > 0,
            F.round(
                F.col("sla_compliant_claims") / F.col("sla_eligible_settled_claims") * 100, 2
            )
        ).otherwise(F.lit(None).cast("double"))
    )
)

# Summary 5: claim_risk_review_summary
claim_risk_review_summary = (
    claim_kpi_base
    .groupBy("risk_band", "review_flag")
    .agg(
        F.countDistinct("claim_id").alias("claim_count"),
        F.sum("requested_amount").alias("requested_amount"),
        F.sum("approved_amount").alias("approved_amount")
    )
)

for name, df in {
    "claim_volume_severity_summary": claim_volume_severity_summary,
    "product_loss_summary": product_loss_summary,
    "provider_performance_summary": provider_performance_summary,
    "claim_processing_sla_summary": claim_processing_sla_summary,
    "claim_risk_review_summary": claim_risk_review_summary
}.items():
    print(f"{name:35s} rows={df.count():>6}")

## 12. KPI spot checks

A KPI is not trustworthy merely because SQL runs. The following checks independently calculate key measures from the claim fact and compare them with the Gold-ready calculation.

In [ ]:
# KPI 1 — Total Claims
total_claims = fact_insurance_claim.select("claim_id").distinct().count()

# KPI 2 — Total Requested Amount
total_requested = fact_insurance_claim.agg(F.sum("requested_amount")).first()[0]

# KPI 3 — Total Approved Amount
total_approved = fact_insurance_claim.agg(F.sum("approved_amount")).first()[0]

# KPI 4 — Claim Approval Rate
adjudicated = fact_insurance_claim.filter(F.col("decision_timestamp").isNotNull()).select("claim_id").distinct().count()
approved = fact_insurance_claim.filter(F.col("decision_timestamp").isNotNull() & (F.col("approved_amount") > 0)).select("claim_id").distinct().count()
approval_rate = (approved / adjudicated * 100) if adjudicated else None

# KPI 5 — Average Claim Severity
approved_amounts = fact_insurance_claim.filter(F.col("approved_amount") > 0)
approved_claims = approved_amounts.select("claim_id").distinct().count()
avg_severity = (float(total_approved) / approved_claims) if approved_claims else None

# KPI 6 — Average Processing Time
processing_avg = (
    claim_kpi_base
    .filter(F.col("submission_timestamp").isNotNull() & F.col("decision_timestamp").isNotNull())
    .agg(F.avg("processing_days"))
    .first()[0]
)

# KPI 7 — Settlement SLA Compliance
sla_row = claim_processing_sla_summary.agg(
    F.sum("sla_compliant_claims").alias("num"),
    F.sum("sla_eligible_settled_claims").alias("den")
).first()
sla_compliance = (float(sla_row["num"]) / sla_row["den"] * 100) if sla_row["den"] else None

# KPI 8 — Outstanding Reserve
# For open approved claims, reserve less cumulative paid, floored at zero.
outstanding_reserve = (
    claim_kpi_base
    .filter(
        F.col("approved_amount") > 0
        & F.col("claim_status").isin("SUBMITTED", "UNDER_REVIEW", "OPEN", "APPROVED")
    )
    .select(
        F.greatest(
            F.col("reserve_amount") - F.col("cumulative_paid_amount"),
            F.lit(0).cast("decimal(18,2)")
        ).alias("outstanding")
    )
    .agg(F.sum("outstanding"))
    .first()[0]
)

kpi_rows = [
    ("Total Claims", total_claims),
    ("Total Requested Amount", total_requested),
    ("Total Approved Amount", total_approved),
    ("Claim Approval Rate %", approval_rate),
    ("Average Claim Severity", avg_severity),
    ("Average Processing Time (days)", processing_avg),
    ("Settlement SLA Compliance %", sla_compliance),
    ("Outstanding Reserve", outstanding_reserve),
]
display(spark.createDataFrame(kpi_rows, ["kpi","value"]))

## 13. Grain and multiplication proof

This is the most important Week 07 proof.

A claim can have many payments. If we join claim rows directly to payment rows, the claim appears once per payment and claim-level amounts are multiplied.

We therefore compare:

- distinct claim count before payment aggregation
- distinct claim count after payment aggregation
- total requested/approved amounts before and after the claim-level payment aggregation

The payment table is reduced to one row per `claim_id` first.

In [ ]:
before_claim_count = fact_insurance_claim.select("claim_id").distinct().count()

after_claim_count = claim_kpi_base.select("claim_id").distinct().count()

before_amounts = fact_insurance_claim.agg(
    F.sum("requested_amount").alias("requested"),
    F.sum("approved_amount").alias("approved")
).first()

after_amounts = claim_kpi_base.agg(
    F.sum("requested_amount").alias("requested"),
    F.sum("approved_amount").alias("approved")
).first()

proof = [
    ("distinct_claims_before", before_claim_count),
    ("distinct_claims_after_safe_payment_aggregation", after_claim_count),
    ("requested_before", float(before_amounts["requested"] or 0)),
    ("requested_after", float(after_amounts["requested"] or 0)),
    ("approved_before", float(before_amounts["approved"] or 0)),
    ("approved_after", float(after_amounts["approved"] or 0)),
]

display(spark.createDataFrame(proof, ["check","value"]))

In [ ]:
grain_pass = before_claim_count == after_claim_count
requested_pass = float(before_amounts["requested"] or 0) == float(after_amounts["requested"] or 0)
approved_pass = float(before_amounts["approved"] or 0) == float(after_amounts["approved"] or 0)

print("Claim grain preserved:", "PASS" if grain_pass else "CHECK")
print("Requested amount preserved:", "PASS" if requested_pass else "CHECK")
print("Approved amount preserved:", "PASS" if approved_pass else "CHECK")

assert grain_pass and requested_pass and approved_pass, "Gold grain/amount reconciliation failed."

## 14. Fact uniqueness validation

Each approved fact must be unique at its business grain:

- policy → `policy_id`
- insurance claim → `claim_id`
- payment → `payment_id`
- status history → `event_id`
- event stream → `event_id`

In [ ]:
fact_unique_checks = {
    "fact_policy": fact_policy.select("policy_id").count() == fact_policy.select("policy_id").distinct().count(),
    "fact_insurance_claim": fact_insurance_claim.select("claim_id").count() == fact_insurance_claim.select("claim_id").distinct().count(),
    "fact_claim_payment": fact_claim_payment.select("payment_id").count() == fact_claim_payment.select("payment_id").distinct().count(),
    "fact_claim_status_history": fact_claim_status_history.select("event_id").count() == fact_claim_status_history.select("event_id").distinct().count(),
    "fact_claim_event_stream": fact_claim_event_stream.select("event_id").count() == fact_claim_event_stream.select("event_id").distinct().count(),
}
display(spark.createDataFrame([(k, "PASS" if v else "CHECK") for k,v in fact_unique_checks.items()], ["fact","status"]))
assert all(fact_unique_checks.values()), "One or more facts are not unique at the approved grain."

## 15. Summary-to-fact reconciliation

Summary counts must reconcile back to their source fact at the same business grain. We use distinct claims for claim-level summaries rather than counting rows after any one-to-many join.

In [ ]:
summary_claim_total = (
    claim_volume_severity_summary
    .agg(F.sum("total_claims").alias("claims"))
    .first()["claims"]
)

fact_claim_total = fact_insurance_claim.select("claim_id").distinct().count()

print("Claim summary total:", summary_claim_total)
print("Fact distinct claims:", fact_claim_total)

# Because the summary is grouped by mutually exclusive claim dimensions,
# its total should equal the distinct claim fact count.
assert int(summary_claim_total) == int(fact_claim_total), "Summary-to-fact claim count mismatch."
print("Summary-to-fact reconciliation: PASS")

## 16. Manual spot-check example

Select one claim and manually trace:

**claim → policy → product → payments aggregated to claim grain**

The purpose is to demonstrate that the payment amount can be inspected without multiplying the claim record.

In [ ]:
anchor = (
    claim_kpi_base
    .select(
        "claim_id", "policy_id", "product_id", "claim_status",
        "requested_amount", "approved_amount", "reserve_amount",
        "cumulative_paid_amount", "processing_days", "settlement_days",
        "sla_target_days"
    )
    .orderBy("claim_id")
    .limit(1)
)

display(anchor)

## 17. Gold output contract

In Databricks, the following objects can be persisted as Delta tables after the Week 6 Trusted Silver dependency is confirmed.

For this local exercise, the dataframes remain notebook objects so the notebook does not create an unintended external catalog/schema.

In [ ]:
gold_objects = {
    **dims,
    **facts,
    "claim_volume_severity_summary": claim_volume_severity_summary,
    "product_loss_summary": product_loss_summary,
    "provider_performance_summary": provider_performance_summary,
    "claim_processing_sla_summary": claim_processing_sla_summary,
    "claim_risk_review_summary": claim_risk_review_summary,
}

for name, df in gold_objects.items():
    print(f"{name:40s} grain-ready rows={df.count():>6}")

## 18. Databricks persistence template

Run this section only after replacing the sample-derived staging layer with the actual Week 6 Trusted Silver tables and confirming the approved catalog/schema.

Example:

```sql
CREATE OR REPLACE TABLE gold.fact_insurance_claim USING DELTA AS
SELECT * FROM fact_insurance_claim;
```

Repeat for the approved dimensions, facts and summaries. Do not publish raw/Bronze/Candidate/quarantine data to Power BI.

## 19. Week 07 evidence checklist

- [x] Ten approved dimensions represented.
- [x] Five approved facts represented.
- [x] Five approved summaries built.
- [x] Eight KPI contracts/calculations represented.
- [x] Fact business-grain uniqueness checked.
- [x] Payment data aggregated to claim grain before claim-level KPI use.
- [x] Summary-to-fact claim reconciliation checked.
- [x] KPI spot-check values generated.
- [ ] Actual Week 6 Trusted Silver reconciliation — must be completed in the internship environment.
- [ ] Delta persistence in the approved Databricks catalog/schema.
- [ ] GitHub commit, Week Log and AI Transparency Note.

**Important limitation:** the uploaded package does not contain the six controlled event-drop files or the completed Week 6 Trusted Silver tables, so this notebook does not invent them. The two event facts are retained as empty contracts and the static source files are used only as sample-derived staging.

## 20. Viva questions — Week 07

1. What is the grain of `fact_insurance_claim`?
2. What is the grain of `fact_claim_payment`?
3. Why must claims and payments remain separate facts?
4. How would a row-level claim-payment join corrupt Total Claims?
5. What is the denominator for Claim Approval Rate?
6. How do you handle a zero denominator?
7. How do you manually spot-check Outstanding Reserve?
8. Why are dimensions built from trusted domains first?
9. What proves a Gold fact is unique at its approved grain?
10. Why should Power BI consume Gold rather than raw, Bronze, Candidate or quarantine?
11. Which calculations can safely use a claim-level payment aggregate?
12. Why is `review_flag` a review indicator rather than proof of fraud?

**Final takeaway:** Build at the approved grain, aggregate finer-grain facts before claim-level analysis, reconcile every important measure, and keep the Gold layer traceable to trusted inputs.